# Q4 — Four simple eco-friendly agents


- **A:** reflex agent for unusual movement in a crowd;
- **B:** utility-based agent for a stressed patient;
- **C:** goal-based agent for a school van route;
- **D:** learning agent for young people's sports activities.

Each program takes a few simple inputs, makes one decision, and prints the result. The data are **made-up examples**. Any energy or water values below are *relative scores for comparing options*, not measured CO₂ emissions or litres of water.

The environmental idea is to avoid needless processing or resource-heavy actions **after basic safety conditions are met**.


## A. Reflex agent — movement in a crowd

A reflex agent follows an immediate **if–then rule**. Its input is a speed estimate and a sudden direction-change estimate supplied by a sensor. If either exceeds a chosen classroom threshold, it alerts a human operator. Otherwise it returns to a low-power waiting mode.

| Part | Example |
|---|---|
| Percept | Speed and direction change |
| Rule | Alert if speed ≥ 8 or change ≥ 90° |
| Action | Alert operator or wait |
| Eco choice | Do simple checks and wait between sensor updates |

The thresholds are illustrative. A movement alert is **not** a conclusion that a person has done anything wrong; a human should review it.


In [ ]:
def crowd_reflex(speed, direction_change):
    if speed >= 8 or direction_change >= 90:
        return "Alert operator"
    else:
        return "No alert; use low-power wait"

crowd_examples = [
    (3, 20),   # ordinary movement
    (9, 30),   # fast movement
    (4, 110),  # sudden turn
]

for speed, change in crowd_examples:
    action = crowd_reflex(speed, change)
    print("Speed:", speed, "Direction change:", change, "->", action)


Speed: 3 Direction change: 20 -> No alert; use low-power wait
Speed: 9 Direction change: 30 -> Alert operator
Speed: 4 Direction change: 110 -> Alert operator


## B. Utility-based agent — stressed or tense patient

A utility-based agent compares possible actions using a score. Here the patient has two **simulated** numbers from 0 to 10: stress and tension. Each action has an illustrative benefit and resource cost. Using an extra room is assigned small energy and cleaning-water costs in this example. The agent chooses the action with the highest score:

$$\text{utility}=\min(\text{stress}+\text{tension},\text{benefit})-\text{energy cost}-\text{water cost}-\text{staff time cost}.$$

The hard safety rule comes first: if stress or tension is 8 or more, notify a staff member. This toy score is **not a clinical assessment or treatment rule**. In practice, qualified staff decide what a patient needs.

| Part | Example |
|---|---|
| Percept | Stress and tension ratings |
| Goal | Offer an appropriate response |
| Utility | Estimated help minus resource and staff costs |
| Eco choice | Prefer a low-resource action when its usefulness is similar |


In [ ]:
actions = [
    # name, possible benefit, energy cost, water cost, staff time cost
    ("Offer a short pause", 3, 0, 0, 0),
    ("Use a quiet room", 7, 1, 1, 0),
    ("Ask staff to help", 9, 0, 0, 4),
]

def patient_agent(stress, tension):
    if stress >= 8 or tension >= 8:
        return "Notify staff now"  # safety rule

    need = stress + tension
    best_action = None
    best_score = -100

    for name, benefit, energy, water, staff_time in actions:
        score = min(need, benefit) - energy - water - staff_time
        if score > best_score:
            best_score = score
            best_action = name

    return best_action

for stress, tension in [(2, 3), (4, 3), (9, 2)]:
    print("Stress:", stress, "Tension:", tension, "->", patient_agent(stress, tension))


Stress: 2 Tension: 3 -> Offer a short pause
Stress: 4 Tension: 3 -> Use a quiet room
Stress: 9 Tension: 2 -> Notify staff now


## C. Goal-based agent — secure school-van commute

A goal-based agent looks for an action that reaches a stated goal. Here the goal is to deliver the registered passengers to every required stop. The program first checks safety conditions: passenger count must fit capacity, boarding must match the registered list, and the route must be marked safe and cover every required stop. **Only then** does it select the route with the smallest distance plus a small idling penalty.

| Part | Example |
|---|---|
| State | Registered/boarded lists, van capacity, candidate routes |
| Goal | Safely cover all required stops |
| Action | Select a route, or hold the van |
| Eco choice | Among safe routes, prefer less travel and idling |

The routes and safety flags below are invented for demonstration. A real service would need verified roads, authorised pickup/release procedures, live conditions, and adult oversight.


In [ ]:
registered = ["Ali", "Sara", "Noor"]
boarded = ["Ali", "Sara", "Noor"]
capacity = 4
required_stops = {"School", "North", "West"}

routes = [
    {"name": "Route A", "stops": {"School", "North", "West"},
     "safe": True, "km": 12, "idle_minutes": 5},
    {"name": "Route B", "stops": {"School", "North", "West"},
     "safe": True, "km": 10, "idle_minutes": 2},
    {"name": "Route C", "stops": {"School", "North", "West"},
     "safe": False, "km": 8, "idle_minutes": 0},
]

def choose_van_route():
    if len(boarded) > capacity or sorted(boarded) != sorted(registered):
        return "Hold van: check passengers"

    best_route = None
    best_cost = 100000
    for route in routes:
        if route["safe"] and route["stops"] == required_stops:
            cost = route["km"] + 0.2 * route["idle_minutes"]
            if cost < best_cost:
                best_cost = cost
                best_route = route["name"]

    if best_route is None:
        return "Hold van: no safe complete route"
    return best_route

for route in routes:
    cost = route["km"] + 0.2 * route["idle_minutes"]
    print(route["name"], "safe:", route["safe"], "travel score:", cost)
print("Chosen action:", choose_van_route())


Route A safe: True travel score: 13.0
Route B safe: True travel score: 10.4
Route C safe: False travel score: 8.0
Chosen action: Route B


## D. Learning agent — encourage sports activities

A learning agent changes its future choices after feedback. This simple agent keeps an interest score for each activity. It recommends the activity with the highest **interest minus resource cost**. After a young person tries it, positive feedback raises that activity's score; negative feedback lowers it.

The resource scores are illustrative. Walking and football need no added water or powered equipment in this example; swimming receives a larger score because pool operation uses resources. The agent does not force an activity or collect personal details. Real recommendations should also consider accessibility, weather, supervision, and the person's preferences.

| Part | Example |
|---|---|
| Percept | Current interest scores and feedback |
| Learning step | Add feedback to an activity's score |
| Action | Recommend an activity |
| Eco choice | Break close choices in favour of lower-resource activities |


In [ ]:
interest = {"Walking": 3, "Football": 2, "Swimming": 4}
resource_cost = {"Walking": 0, "Football": 0, "Swimming": 2}

def recommend_activity():
    best_activity = None
    best_score = -100

    for activity in interest:
        score = interest[activity] - resource_cost[activity]
        if score > best_score:
            best_score = score
            best_activity = activity

    return best_activity

def learn_from_feedback(activity, liked_it):
    if liked_it:
        interest[activity] += 1
    else:
        interest[activity] -= 2

first_choice = recommend_activity()
print("First recommendation:", first_choice)

# Example feedback: the person did not enjoy the first suggestion.
learn_from_feedback(first_choice, False)
print("Scores after feedback:", interest)
print("Next recommendation:", recommend_activity())


First recommendation: Walking
Scores after feedback: {'Walking': 1, 'Football': 2, 'Swimming': 4}
Next recommendation: Football


## Summary

| Agent | How it decides | How it avoids unnecessary resource use |
|---|---|---|
| Reflex | Fixed if–then rules | Checks simple sensor values; waits at low power when normal |
| Utility-based | Chooses highest score after a safety check | Includes energy, water, and staff costs in the score |
| Goal-based | Satisfies safety and stop goals, then compares routes | Prefers shorter routes with less idling |
| Learning | Updates preferences from feedback | Includes a resource cost when choosing activities |
